# Global Health Econometric Analysis & EDA

Runs the calibrated synthetic data generator, bivariate/multivariate OLS regressions, VIF multicollinearity checks, Shapiro-Wilk residual diagnostics, and IEEE PDF report builder.

In [ ]:
import os
from pathlib import Path

if not Path("src").exists():
    !git clone https://github.com/GaneshArwan/global_health_eda.git
    os.chdir("global_health_eda")
    !pip install -q -r requirements.txt

print("Working directory:", Path.cwd())

In [ ]:
import pandas as pd
from src.data_generator import generate_global_health_data
from src.guardrails import resolve_safe_path, validate_health_dataframe
from src.generate_eda_report import compute_econometric_summary

csv_path = resolve_safe_path("data/global_health_data.csv")
csv_path.parent.mkdir(parents=True, exist_ok=True)
raw_df = generate_global_health_data(num_countries=200, random_seed=42)
raw_df.to_csv(csv_path, index=False)

clean_df = validate_health_dataframe(raw_df)
missing_counts = {str(k): int(v) for k, v in raw_df.isna().sum().items() if v > 0}
stats = compute_econometric_summary(clean_df, initial_rows=len(raw_df), missing_counts=missing_counts)

print(f"Complete cases: {stats['n_obs']}/{stats['initial_rows']} ({stats['retention_rate']:.1f}%)")
print("Descriptive stats:")
display(pd.DataFrame(stats["descriptive"])[["label", "mean", "std", "median", "iqr", "min", "max"]].round(2))
print("Regional summary:")
display(pd.DataFrame(stats["regional"]).round(2))

In [ ]:
m1, m2, m3 = stats["model1"], stats["model2"], stats["model3"]
print(f"Model 1 (Preston Curve ln(GDP) -> Life_Expectancy): R2={m1['r2']:.3f}, slope={m1['slope']:.2f}, RMSE={m1['rmse']:.2f}, p={m1['p']:.2e}")
print(f"Shapiro-Wilk residuals: W={m1['shapiro_w']:.3f}, p={m1['shapiro_p']:.3f}; outliers(|Z|>2)={m1['outliers_count']}")
print(f"Model 2 (Education -> Child_Mortality): R2={m2['r2']:.3f}, slope={m2['slope']:.1f}, RMSE={m2['rmse']:.2f}, p={m2['p']:.2e}")
print(f"Model 3 (Multivariate OLS): R2={m3['r2']:.3f}, Adj. R2={m3['adj_r2']:.3f}, RMSE={m3['rmse']:.2f}, Max VIF={m3['max_vif']:.2f}")
display(pd.DataFrame(m3["predictors"]).round(4))

In [ ]:
from IPython.display import Image as IPyImage, display
from src.generate_eda_report import generate_eda_report

generate_eda_report()
for img_name in ["preston_curve.png", "edu_mort.png", "heatmap.png", "residuals_diagnostics.png"]:
    display(IPyImage(filename=f"temp_assets/{img_name}"))